# Business Entity Resolution: Kaggle Execution Runner

Shard-streaming architecture with hard RAM ceiling (~8 GB peak):
- TF-IDF vocabulary fitted on a 500K sample, never on full corpus
- Candidates streamed in 300K-row shards, never held fully in memory
- Filtered candidate reload: only loads IDs selected by blocking
- Aggressive gc.collect() between every stage
- Live unbuffered streaming output via `python -u`
- Incremental checkpointing


### Step 1: Clone Repository & Pull Latest Code


In [ ]:
import os, sys

GITHUB_REPO_URL = "https://github.com/purvanshjoshi/business-entity-resolution.git"

if not os.path.exists('run_pipeline.py'):
    if not os.path.exists('business-entity-resolution'):
        print("Cloning repository from GitHub...")
        os.system(f"git clone {GITHUB_REPO_URL}")

    if os.path.exists('business-entity-resolution/code/business_entity_resolution'):
        os.chdir('business-entity-resolution/code/business_entity_resolution')
        print("Changed directory to business-entity-resolution/code/business_entity_resolution/")

# Always pull the latest code
os.system("git pull origin main")
print(f"Current working directory: {os.getcwd()}")
print("Directory contents:", os.listdir('.'))


### Step 2: Install Dependencies


In [ ]:
!pip install -q -r requirements.txt
print("Dependencies verified!")


### Step 3: Run Full Pipeline (Shard-Streaming Architecture)

Key parameters:
- `--shard-size 300000`: Stream candidates in 300K-row shards (never holds full matrix)
- `--batch-size 1000`: S1 sub-batch size for sparse multiply
- `--max-features 50000`: TF-IDF vocabulary cap (50K vs previous 150K = 3x less RAM)
- `--top-k 12`: Top candidates per entity
- `--sample-train 100000`: Training sample size


In [ ]:
!python -u run_pipeline.py \
    --output-dir /kaggle/working/output \
    --sample-train 100000 \
    --top-k 12 \
    --batch-size 1000 \
    --shard-size 300000 \
    --max-features 50000


### Step 4: Verify Output Files


In [ ]:
import os
import pandas as pd

output_dir = '/kaggle/working/output' if os.path.exists('/kaggle/working/output') else './output'
matching_file = os.path.join(output_dir, 'matching_results.tsv')
candidate_file = os.path.join(output_dir, 'candidate_pairs.tsv')

print("=" * 60)
print("OUTPUT VERIFICATION:")
print("=" * 60)

if os.path.exists(matching_file):
    df_match = pd.read_csv(matching_file, sep='\t')
    print(f"matching_results.tsv exists! Total rows: {len(df_match):,}")
    print("Preview:")
    print(df_match.head(10))
else:
    print("matching_results.tsv not found!")

if os.path.exists(candidate_file):
    df_cand = pd.read_csv(candidate_file, sep='\t')
    print(f"\ncandidate_pairs.tsv exists! Total rows: {len(df_cand):,}")
    print("Preview:")
    print(df_cand.head(5))
else:
    print("candidate_pairs.tsv not found!")

print("=" * 60)
print(f"Ready to submit: {matching_file}")
print("=" * 60)
